In [ ]:
from AlgorithmImports import (    QCAlgorithm, Resolution, BrokerageName, AccountType,    OrderEvent, OrderStatus, ConstantSlippageModel, PortfolioTarget)from collections import dequeimport json, mathfrom datetime import datetime, timezone, timedeltaclass ExternalSignalConsumer(QCAlgorithm):    """    Consumes model signals from a JSON endpoint and maps them to long-only weights.    Improvements vs prior:      • Sizing modes to avoid flat ~0.5 leverage (linear or thresholded).      • Rolling 63d beta vs SPY.      • Daily Sharpe/PSR plots & end-of-run summary (orders, fills, win rate).    """    def Initialize(self) -> None:        # Backtest window        self.SetStartDate(2024, 1, 1)        self.SetEndDate(2024, 12, 31)        self.SetCash(100000)        self.mode           = (self.GetParameter("Mode") or "json-live").strip().lower()        self.json_url       = (self.GetParameter("SignalsUrl") or "").strip()        self.symbols_param  = (self.GetParameter("Symbols") or "UNH,TSLA,TMO").strip()        self.poll_minutes   = int(self.GetParameter("PollingMinutes") or 60)        self.sizing_mode     = (self.GetParameter("SizingMode") or "threshold").strip().lower()        self.w_cap           = float(self.GetParameter("WeightCap") or 0.6)        self.conf_floor      = float(self.GetParameter("ConfidenceFloor") or 0.60)        self.ThrowIfEmptyUrl("SignalsUrl")        self.symbols = {}        for tkr in [s.strip().upper() for s in self.symbols_param.split(",") if s.strip()]:            self.symbols[tkr] = self.AddEquity(tkr, Resolution.HOUR).Symbol        self.SetSecurityInitializer(lambda sec: sec.SetSlippageModel(ConstantSlippageModel(0.01)))        self.SetBrokerageModel(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE, AccountType.MARGIN)        self.UniverseSettings.Leverage = 1.0        self.Settings.FreePortfolioValuePercentage = 0.05        self.Settings.MinimumOrderMarginPortfolioPercentage = 0.0        self.Settings.RebalancePortfolioOnSecurityChanges = False        self.Schedule.On(            self.DateRules.EveryDay(),            self.TimeRules.Every(timedelta(minutes=self.poll_minutes)),            self.PollJsonAndTrade        )        # Metrics / state        self.fill_count = 0        self.last_equity = float(self.Portfolio.TotalPortfolioValue)        self.daily_returns = []        self.valid_until_epoch = None        self.last_poll_minute = None        self.last_logged_sig = {}        self.SetWarmup(5, Resolution.HOUR)        self.spy = self.AddEquity("SPY", Resolution.DAILY).Symbol        self.Schedule.On(            self.DateRules.EveryDay(self.spy),            self.TimeRules.AfterMarketClose(self.spy, 1),            self.PushDailyMetrics        )        self.bench = self.spy        self.SetBenchmark(self.bench)        self._p_rets = deque(maxlen=63)        self._b_rets = deque(maxlen=63)        self._last_bench_close = None        self._last_eod_date = None        self._order_ids = set()        self._fills = 0        self.Log(f"Params OK | mode={self.mode} poll={self.poll_minutes} sizing={self.sizing_mode} "                 f"cap={self.w_cap:.2f} floor={self.conf_floor:.2f} url={self.json_url[:60]}...")